Character Level Language Model

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(len(text))

1115394


In [3]:
chars = sorted(list(set(text)))  # to get all characters there are!
vocal_size = len(chars)
print(''.join(chars))
vocal_size


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


65

In [4]:
# a simple mapping from characters to integers[0-64]
stoi = {ch : i for i, ch in enumerate(chars)}
itos = {i : ch for i, ch in enumerate(chars)}
encode = lambda s : [stoi[c] for c in s]
decode = lambda l : ''.join([itos[i] for i in l])
print(encode("hii  there"))
print(decode(encode("hii  there")))

# encode the dataset
data = torch.tensor(encode(text))
print(data.shape, data.dtype)
#print(data[:1000])

[46, 47, 47, 1, 1, 58, 46, 43, 56, 43]
hii  there
torch.Size([1115394]) torch.int64


In [5]:
# spliting into train and val sets
n = int(0.9 * len(data))
train_set = data[:n]
val_set = data[n:]
print(len(train_set), len(val_set))

1003854 111540


In [6]:
batch_size = 4 #how many independent sequences of block we process in parallel
block_size = 8 #max context length for prediction

def get_batch(split):
    data = train_set if split == 'train' else val_set
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i+1 : i + block_size + 1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(xb.shape, yb.shape)  # torch.Size([4, 8]) as 4 batches of 8 blocks run in parallel!!


torch.Size([4, 8]) torch.Size([4, 8])


In [7]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocal_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocal_size, vocal_size) #each token reads from this table for the next token

    def forward(self, idx, targets=None):
        #(B, T, C) idx and targets are (B, T)
        logits = self.token_embedding_table(idx)   # the next token prediction 
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)               # as cross_entropy expects input (minibatch, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)               # get the next prediction
            logits = logits[:, -1, :] # as its a bigram only see the last char to predict next one
            probs = F.softmax(logits, dim=1)        # get prob for every char
            idx_next = torch.multinomial(probs, num_samples=1)  # we get one sample from probs at random (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)
        return idx


m = BigramLanguageModel(vocal_size)
logits, loss = m(xb, yb)
print(logits.shape, loss)
idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx, 120)[0].tolist()))   # as generate returns (B, T+1) we want to get into the first row for the first row prediction!


torch.Size([32, 65]) tensor(4.5822, grad_fn=<NllLossBackward0>)

nuSN:iSnG.WN QjS?wdcTvK;jHZCKDqlaWgpQ;u-M& aC',axAC,'uRSqsp.LYM.;N;;
$h$'?wBy,RQDsHCd?wxNKGk3XLLQhjZrDdvLQ-U yV?kHdGf,l,


In [ ]:
@torch.no_grad()
def estimate_loss():
    m.eval()
    out = {}
    for split in ['train' , 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            xb, yb = get_batch(split)
            logits, loss = m(xb, yb)
            losses[k] = loss.item()

        out[split] = losses.mean()
    m.train()
    return out

In [9]:
# training the model
batch_size = 32
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)

for i in range(10000):
    if i % 500 == 0:
        losses = estimate_loss()
        print(
            f"step {i}: "
            f"train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    opt.zero_grad()
    loss.backward()
    opt.step()

step 0: train loss 4.6464, val loss 4.6568
step 500: train loss 4.1180, val loss 4.1360
step 1000: train loss 3.6957, val loss 3.7225
step 1500: train loss 3.3802, val loss 3.3972
step 2000: train loss 3.1317, val loss 3.1505
step 2500: train loss 2.9464, val loss 2.9594
step 3000: train loss 2.8040, val loss 2.8238
step 3500: train loss 2.7106, val loss 2.7252
step 4000: train loss 2.6349, val loss 2.6543
step 4500: train loss 2.5866, val loss 2.6123
step 5000: train loss 2.5547, val loss 2.5741
step 5500: train loss 2.5293, val loss 2.5484
step 6000: train loss 2.5153, val loss 2.5406
step 6500: train loss 2.5092, val loss 2.5269
step 7000: train loss 2.4922, val loss 2.5109
step 7500: train loss 2.4877, val loss 2.5026
step 8000: train loss 2.4735, val loss 2.4939
step 8500: train loss 2.4800, val loss 2.4870
step 9000: train loss 2.4798, val loss 2.4733
step 9500: train loss 2.4690, val loss 2.4847


In [14]:
print(decode(m.generate(idx, 420)[0].tolist()))


Amed hyousousto wf ous sheno me thy h s an d y gisspicound t'd:
Whe, t rthis t is INoBRIUCKe pr, outht f
SwGAnthave my bl tepolll mait G wang cen thar,
IAn'stons ckn gheonghiot ARolyr:

thal ted d tt an keave tR:
PEO: in rmelodeghos st ashoueflicoupen'that d ored,
The itrenosh CUCENUCLe'scol thiteno win fang
STI l wUCH:
Gond
s 'sh o y,
donk'd
T:

D!
RAHESTorere om
Theha basthe atapthe padeveime n, GUTread, drkngun, 3
